# Day 078 — Exercise 5: process_media and MediaStudio

**What you'll build:** The routing layer and the capstone class.

**Why it matters:** `process_media` makes the studio file-type-aware. `MediaStudio` is the Section 5 capstone class — 13 injectable classes, all following the same bind-at-construction pattern.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(w=100, h=100, color=(100, 150, 200)):
    return _PILImage.new('RGB', (w, h), color=color)

_mock_describe_fn   = lambda img, q: 'A test image with a solid color background.'
_mock_transcribe_fn = lambda src: {'text': 'Hello world.', 'segments': [
    {'start': 0.0, 'end': 1.0, 'text': 'Hello world.'}]}
_mock_tts_fn        = lambda text, voice, rate, pitch: b'AUDIO:' + text[:8].encode()
from pathlib import Path

MEDIA_EXTENSIONS = {
    'image': {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff'},
    'audio': {'.mp3', '.wav', '.ogg', '.flac', '.m4a', '.aac'},
    'video': {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'},
}

def detect_media_type(path):
    ext = Path(path).suffix.lower()
    for media_type, extensions in MEDIA_EXTENSIONS.items():
        if ext in extensions:
            return media_type
    return 'unknown'
import io, base64

def describe_media(source, describe_fn=None):
    from PIL import Image
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    else:
        image = source
    prompt = 'Describe this image in detail, including all visible content.'
    if describe_fn is not None:
        return describe_fn(image, prompt)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}],
    )
    return resp['message']['content']
def transcribe_media(source, transcribe_fn=None):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper, os, tempfile
    model = whisper.load_model('base')
    if isinstance(source, bytes):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            raw = model.transcribe(tmp)
        finally:
            os.unlink(tmp)
    else:
        raw = model.transcribe(str(source))
    segments = [
        {'start': s['start'], 'end': s['end'], 'text': s['text'].strip()}
        for s in raw.get('segments', [])
    ]
    return {'text': raw.get('text', '').strip(), 'segments': segments}
import asyncio

def synthesize_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz',
                      tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import edge_tts
    async def _run():
        chunks = []
        async for chunk in edge_tts.Communicate(
                text, voice, rate=rate, pitch=pitch).stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

def narrate_image(source, voice='en-US-AriaNeural', describe_fn=None, tts_fn=None):
    description = describe_media(source, describe_fn=describe_fn)
    audio = synthesize_speech(description, voice=voice, tts_fn=tts_fn)
    return {'description': description, 'audio': audio}


## Task

1. `process_media(source, describe_fn=None, transcribe_fn=None, tts_fn=None) -> dict`
   - `media_type = detect_media_type(source)`
   - `if image`: `result = describe_media(source, describe_fn=describe_fn)`
   - `elif audio`: `result = transcribe_media(source, transcribe_fn=transcribe_fn)`
   - `else`: `result = {'note': f'Media type {media_type!r} detected but not processed'}`
   - Return `{'type': media_type, 'source': str(source), 'result': result}`

2. `MediaStudio(describe_fn=None, transcribe_fn=None, tts_fn=None)`
   - Store all 3 injections
   - `describe/transcribe/speak/narrate/process`: one delegation line each
   - `batch(sources)`: `[self.process(s) for s in sources]`

## Your Implementation

In [ ]:
def process_media(source, describe_fn=None, transcribe_fn=None, tts_fn=None):
    """Auto-detect media type and process. Returns {type, source, result}."""
    raise NotImplementedError

class MediaStudio:
    """Multimodal media processing studio."""

    def __init__(self, describe_fn=None, transcribe_fn=None, tts_fn=None):
        raise NotImplementedError

    def describe(self, source):
        raise NotImplementedError

    def transcribe(self, source):
        raise NotImplementedError

    def speak(self, text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz'):
        raise NotImplementedError

    def narrate(self, source, voice='en-US-AriaNeural'):
        raise NotImplementedError

    def process(self, source):
        raise NotImplementedError

    def batch(self, sources):
        raise NotImplementedError


In [ ]:
def process_media(source, describe_fn=None, transcribe_fn=None, tts_fn=None):
    media_type = detect_media_type(source)
    if media_type == 'image':
        result = describe_media(source, describe_fn=describe_fn)
    elif media_type == 'audio':
        result = transcribe_media(source, transcribe_fn=transcribe_fn)
    else:
        result = {'note': f'Media type {media_type!r} detected but not processed'}
    return {'type': media_type, 'source': str(source), 'result': result}

class MediaStudio:
    def __init__(self, describe_fn=None, transcribe_fn=None, tts_fn=None):
        self._describe_fn = describe_fn
        self._transcribe_fn = transcribe_fn
        self._tts_fn = tts_fn

    def describe(self, source):
        return describe_media(source, describe_fn=self._describe_fn)

    def transcribe(self, source):
        return transcribe_media(source, transcribe_fn=self._transcribe_fn)

    def speak(self, text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz'):
        return synthesize_speech(text, voice=voice, rate=rate, pitch=pitch,
                                 tts_fn=self._tts_fn)

    def narrate(self, source, voice='en-US-AriaNeural'):
        return narrate_image(source, voice=voice,
                             describe_fn=self._describe_fn, tts_fn=self._tts_fn)

    def process(self, source):
        return process_media(source, describe_fn=self._describe_fn,
                             transcribe_fn=self._transcribe_fn, tts_fn=self._tts_fn)

    def batch(self, sources):
        return [self.process(s) for s in sources]


## Automated checks

In [ ]:

score, total = 0, 6
try:
    from PIL import Image as PILImage
    import tempfile, os

    # process_media with image
    img = _make_mock_image()
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        img.save(f, format='PNG'); tmp_img = f.name
    try:
        r = process_media(tmp_img, describe_fn=_mock_describe_fn)
        assert r['type'] == 'image' and isinstance(r['result'], str)
        score += 1; print("✅ process_media dispatches image to describe_media")
    finally:
        os.unlink(tmp_img)

    # process_media with audio (file need not exist — mock)
    r2 = process_media('test.mp3', transcribe_fn=_mock_transcribe_fn)
    assert r2['type'] == 'audio' and isinstance(r2['result'], dict)
    score += 1; print("✅ process_media dispatches audio to transcribe_media")

    # process_media with unknown
    r3 = process_media('data.csv')
    assert r3['type'] == 'unknown' and 'note' in r3['result']
    score += 1; print("✅ process_media handles unknown type gracefully")

    # MediaStudio describe
    studio = MediaStudio(describe_fn=_mock_describe_fn,
                         transcribe_fn=_mock_transcribe_fn,
                         tts_fn=_mock_tts_fn)
    d = studio.describe(PILImage.new('RGB', (10, 10)))
    assert isinstance(d, str)
    score += 1; print("✅ MediaStudio.describe returns str")

    t = studio.transcribe(b'audio')
    assert isinstance(t, dict) and 'text' in t
    score += 1; print("✅ MediaStudio.transcribe returns {text, segments}")

    results = studio.batch(['photo.png', 'clip.mp4', 'notes.txt'])
    assert len(results) == 3 and all('type' in r for r in results)
    score += 1; print("✅ MediaStudio.batch returns list of 3 result dicts")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def process_media(source, describe_fn=None, transcribe_fn=None, tts_fn=None):
    media_type = detect_media_type(source)
    if media_type == 'image':
        result = describe_media(source, describe_fn=describe_fn)
    elif media_type == 'audio':
        result = transcribe_media(source, transcribe_fn=transcribe_fn)
    else:
        result = {'note': f'Media type {media_type!r} detected but not processed'}
    return {'type': media_type, 'source': str(source), 'result': result}

class MediaStudio:
    def __init__(self, describe_fn=None, transcribe_fn=None, tts_fn=None):
        self._describe_fn = describe_fn
        self._transcribe_fn = transcribe_fn
        self._tts_fn = tts_fn

    def describe(self, source):
        return describe_media(source, describe_fn=self._describe_fn)

    def transcribe(self, source):
        return transcribe_media(source, transcribe_fn=self._transcribe_fn)

    def speak(self, text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz'):
        return synthesize_speech(text, voice=voice, rate=rate, pitch=pitch,
                                 tts_fn=self._tts_fn)

    def narrate(self, source, voice='en-US-AriaNeural'):
        return narrate_image(source, voice=voice,
                             describe_fn=self._describe_fn, tts_fn=self._tts_fn)

    def process(self, source):
        return process_media(source, describe_fn=self._describe_fn,
                             transcribe_fn=self._transcribe_fn, tts_fn=self._tts_fn)

    def batch(self, sources):
        return [self.process(s) for s in sources]
```

**Why `{'note': ...}` instead of `raise ValueError` for unknown types?** In a batch pipeline, one unsupported file should not crash the whole job. Returning a descriptive dict lets the caller decide how to handle it.

</details>